## Training configuration
- **Epochs**: 60 with early stopping (patience=8)
- **Batch size**: 16 for better gradient estimates
- **Augmentation**: crop, flip, rotate, jitter, erase
- **Scheduler**: 5-epoch warmup → cosine annealing
- **Label smoothing**: 0.1 to reduce overconfidence
- **TTA at inference**: 8 augmented passes averaged

Expected improvement over baseline:
- Stage 1 accuracy: ~85% → ~92%+
- Stage 2 accuracy: ~70% → ~80%+

In [1]:
import sys
sys.path.append('..')
import torch
from src.dataset import get_tumor_dataloaders
from src.model   import build_tumor_classifier
from src.train   import train_model, plot_history
from config      import *

print(f" Imports done | Device: {DEVICE}")

Using device: cpu
 Imports done | Device: cpu


In [2]:
train_loader, val_loader, test_loader = \
    get_tumor_dataloaders(batch_size=BATCH_SIZE)

Loaded 5600 images from e:\Collage_submission\NeuroVision-AI\notebooks\..\data\tumor_type\Training
Loaded 1600 images from e:\Collage_submission\NeuroVision-AI\notebooks\..\data\tumor_type\Testing

DataLoaders ready:
  Train : 5600 images
  Val   : 800 images
  Test  : 800 images


In [3]:
import numpy as np
import torch
from src.dataset import TumorDataset

train_ds     = TumorDataset(TRAIN_DIR, TUMOR_CLASSES)
counts       = train_ds.get_class_counts()
total        = sum(counts)
weights      = [total / (len(counts) * c) for c in counts]
class_weights = torch.tensor(weights, dtype=torch.float32)

print("Class weights:")
for cls, w in zip(TUMOR_CLASSES, weights):
    print(f"  {cls:12s}: {w:.4f}")

Loaded 5600 images from e:\Collage_submission\NeuroVision-AI\notebooks\..\data\tumor_type\Training
Class weights:
  glioma      : 1.0000
  meningioma  : 1.0000
  notumor     : 1.0000
  pituitary   : 1.0000


In [4]:
model = build_tumor_classifier(
    num_classes=NUM_CLASSES,
    pretrained=True
)

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to C:\Users\tbans/.cache\torch\hub\checkpoints\efficientnet_b2_rwightman-c35c1473.pth
100%|██████████| 35.2M/35.2M [00:02<00:00, 13.5MB/s]


Model     : EfficientNet-B2
Total params    : 8,424,454
Trainable params: 3,957,712 (47.0%)


In [ ]:
history = train_model(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = EPOCHS,
    lr           = LEARNING_RATE,
    save_path    = MODELS_DIR / 'tumor_classifier.pth',
    class_weights= class_weights,
)


  Training on cpu | Epochs: 30



Epoch 01/30 | Train Loss: 0.6861 Acc: 76.57% | Val Loss: 0.5169 Acc: 81.75%  ✅ Saved (best)


Epoch 02/30 | Train Loss: 0.3291 Acc: 87.61% | Val Loss: 0.7027 Acc: 85.88%  ✅ Saved (best)


Epoch 03/30 | Train Loss: 0.2666 Acc: 89.98% | Val Loss: 1.5865 Acc: 88.00%  ✅ Saved (best)


Epoch 04/30 | Train Loss: 0.2188 Acc: 91.57% | Val Loss: 0.3451 Acc: 90.62%  ✅ Saved (best)


Epoch 05/30 | Train Loss: 0.1904 Acc: 92.93% | Val Loss: 0.5764 Acc: 88.25%


Epoch 06/30 | Train Loss: 0.1786 Acc: 93.71% | Val Loss: 0.4455 Acc: 88.75%


train:  24%|██▍       | 42/175 [05:31<13:47,  6.22s/it] 

In [ ]:
plot_history(history,
             title='Tumor Classifier — Training',
             save_name='tumor_training_curves.png')

In [ ]:
from src.train import run_epoch
import torch.nn as nn

# Load best checkpoint
ckpt  = torch.load(MODELS_DIR / 'tumor_classifier.pth',
                   map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} "
      f"| Val acc: {ckpt['val_acc']*100:.2f}%")

criterion      = nn.CrossEntropyLoss(label_smoothing=0.1)
test_loss, test_acc = run_epoch(model, test_loader,
                                criterion)
print(f"\nTest Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc*100:.2f}%")